# GiTrust × Kev-0.8B — fine-tune on Colab, serve locally

Trains a LoRA delta on top of the released **Kev-0.8B v1.0** for GiTrust's Layer-1 routing question, then packages the
run folder for download. Inference stays on the laptop (`kev.serve --run runs/gitrust-v1`); only the synthetic training
phrasings are uploaded here (no code, no diffs).

**Runtime → Change runtime type → GPU** (a free T4 is enough). Then run the cells top to bottom (~20–30 min on a T4).

Upload when prompted: `kev_train.jsonl` and `kev_dev.jsonl` from `Glit/docs/classifier_eval/`.
The 44-case `eval_set.json` is **not** uploaded on purpose; it's the held-out test, scored on the laptop.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("bf16 supported:", torch.cuda.is_bf16_supported())

## 1 · Install Kev (pinned to the commit tested locally)

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os; os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!git clone -q https://github.com/jaredpalmer/kev.git /content/kev
%cd /content/kev
!git checkout -q 85ccf56a0afb952323c621055b02876e8498d60c && git log -1 --oneline
!uv sync --extra serve 2>&1 | tail -3

## 2 · Upload the training data

In [ ]:
from google.colab import files
up = files.upload()   # pick kev_train.jsonl and kev_dev.jsonl
assert {"kev_train.jsonl", "kev_dev.jsonl"} <= set(up), up.keys()
!wc -l kev_train.jsonl kev_dev.jsonl

## 3 · Train

* `--init_from jaredpalmer/kev-0.8b@v1.0` — delta mode: keeps the released model's skill, adapts to GiTrust's ops.
* `--perm_kl 1.0` — penalises answers that change when the options are reordered (the stock model flipped on 18/44).
* fp32 unless the GPU does bf16 (a T4 does not); `--checkpointing 1` keeps activations small.

In [ ]:
import torch, subprocess
dtype = "bf16" if torch.cuda.is_bf16_supported() else "fp32"
cmd = ("uv run python -m kev.train --data kev_train.jsonl --base Qwen/Qwen3.5-0.8B-Base "
       "--init_from jaredpalmer/kev-0.8b@v1.0 --epochs 2 --lr 2e-5 --batch 1 --accum 8 "
       f"--checkpointing 1 --dtype {dtype} --device cuda --perm_kl 1.0 --seed 0 --out runs/gitrust-v1")
print(cmd)
!{cmd} 2>&1 | grep -v "it/s" | tail -40
!ls -la runs/gitrust-v1

## 4 · Sanity check on the dev split (stock vs fine-tuned)

Serves each checkpoint on the GPU and scores `kev_dev.jsonl` (phrasings never trained on). This is only a check that
training helped; the real test is the untouched 44-case set on the laptop. If serving fails on this GPU, skip to step 5;
the laptop eval covers it.

In [ ]:
import json, os, signal, subprocess, time, urllib.request

def models(port):
    return json.loads(urllib.request.urlopen(f"http://127.0.0.1:{port}/v1/models", timeout=2).read())["models"][0]

def port_free(port):
    try:
        models(port); return False
    except Exception:
        return True

def serve(run, port):
    # `uv run` spawns python as a CHILD: killing only `uv` leaves the server on
    # the port, and the next "serve" would silently score the OLD model. So:
    # own process group, kill the group, and verify which run is answering.
    assert port_free(port), f"port {port} already in use — run: !pkill -f kev.serve"
    env = dict(os.environ, KEV_DTYPE="fp32")
    p = subprocess.Popen(["uv", "run", "--extra", "serve", "python", "-m", "kev.serve", "--run", run, "--port", str(port)],
                         env=env, stdout=open(f"serve-{port}.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(180):
        try:
            m = models(port)
            assert m["run"].rstrip("/").endswith(run.rstrip("/")), f"wrong model answering: {m['run']} (expected {run})"
            print("serving", m["run"]); return p
        except AssertionError:
            stop(p); raise
        except Exception:
            time.sleep(2)
    stop(p); raise RuntimeError(open(f"serve-{port}.log").read()[-2000:])

def stop(p):
    try: os.killpg(p.pid, signal.SIGTERM)
    except ProcessLookupError: pass
    p.wait()
    for _ in range(30):
        if port_free(8009): return
        time.sleep(1)
    raise RuntimeError("server did not stop")

def score(port):
    ok = n = 0
    for line in open("kev_dev.jsonl"):
        r = json.loads(line); q = r["questions"]["op"]; gold = q.pop("label")
        body = {"model": "kev-latest", "state": r["state"], "questions": {"op": q}}
        req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/systemone", data=json.dumps(body).encode(),
                                     headers={"content-type": "application/json"})
        a = json.loads(urllib.request.urlopen(req, timeout=120).read())["answers"]["op"]
        ok += a["choice"] == gold; n += 1
    return ok, n

for name, run in [("stock", "jaredpalmer/kev-0.8b@v1.0"), ("fine-tuned", "runs/gitrust-v1")]:
    p = serve(run, 8009)
    try:
        ok, n = score(8009); print(f"{name:11} dev accuracy {ok}/{n} = {100*ok/n:.1f}%")
    finally:
        stop(p)


## 5 · Download the run folder

Unzip it into `~/Desktop/personal/kev/runs/` on the laptop.

In [ ]:
!cd runs && zip -qr /content/gitrust-v1.zip gitrust-v1 && ls -la /content/gitrust-v1.zip
files.download("/content/gitrust-v1.zip")